In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.calibration import (
    to_long_format,
    fit_all_calibration_curves,
    calibrate_matches,
)
from transforms.elo import build_elo
from transforms.scoring import build_mart_coupon

SILVER_TABLE = "stryktipset.silver.matches"
COUPON_TABLE = "stryktipset.mart.coupon"

In [ ]:
def load_matches(table: str = SILVER_TABLE) -> DataFrame:
    """Silver, unscoped -- every league and country, played and not."""
    return spark.table(table)


def open_draw_number(matches: DataFrame) -> int:
    """The newest draw in silver.

    01 fetches the open draw now, so this is normally the coupon that has
    not been played yet. Once its matches finish it is simply the most
    recent completed one, and scoring it again is harmless.
    """
    return matches.agg(F.max("draw_number")).first()[0]

In [ ]:
def write_coupon(df: DataFrame, table: str = COUPON_TABLE) -> None:
    """Create the table on first run, then MERGE on match_id.

    A coupon is scored repeatedly through the week -- streck keeps moving
    until the draw closes -- so re-running must update a row in place
    rather than append a duplicate. The wildcard UPDATE SET * / INSERT *
    forms are used because those are the ones Delta's schema evolution
    actually supports.
    """
    if not spark.catalog.tableExists(table):
        df.write.format("delta").saveAsTable(table)
        print(f"created {table}")
        return

    source_view = "coupon_source"
    df.createOrReplaceTempView(source_view)
    spark.sql(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {table} AS target
        USING {source_view} AS source
        ON target.match_key = source.match_key
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [ ]:
def show(rows) -> str:
    """Print the coupon the way you would read it on a Thursday, and return
    a compact one-line form for dbutils.notebook.exit so a run can be read
    from the Jobs API without opening the notebook.

    The `gap` column compares like with like, which needs care.
    elo_expected_score is an *expected score* on Elo's chess scale, where a draw
    counts as half a win -- it is not P(home win). Subtracting streck_1
    from it would bake in roughly half the draw probability and make the
    crowd look systematically wrong on every single match.

    So the crowd is converted to the same scale first:

        crowd expected score = streck_1 + streck_x / 2

    gap is then that minus elo_expected_score. Positive means the crowd rates the
    home side higher than the ratings do; negative the reverse. Rows near
    zero agree and tell you nothing you didn't already know.
    """
    header = f"{'#':<3}{'match':<34}{'league':<20}{'streck 1/X/2':<18}{'elo xs':>7}{'gap':>7}"
    print(header)
    print("-" * len(header))

    compact = []
    for row in rows:
        fixture = f"{row['home_team']} - {row['away_team']}"
        league = row["league"] or ""
        streck_1 = row["streck_1"]
        elo_1 = row["elo_expected_score"]

        if streck_1 is None:
            streck, crowd_score = "-", None
        else:
            streck = "{:.0%}/{:.0%}/{:.0%}".format(
                streck_1, row["streck_x"], row["streck_2"]
            )
            # same scale as Elo: a draw is half a win
            crowd_score = streck_1 + row["streck_x"] / 2
        elo = "-" if elo_1 is None else "{:.0%}".format(elo_1)
        if crowd_score is None or elo_1 is None:
            gap = "-"
        else:
            gap = "{:+.0%}".format(crowd_score - elo_1)

        print(f"{row['event_number']:<3}{fixture[:33]:<34}{league[:19]:<20}{streck:<18}{elo:>6}{gap:>7}")
        compact.append(f"{row['event_number']}:{fixture[:22]} s={streck} e={elo} gap={gap}")

    return " | ".join(compact)

In [ ]:
def main():
    """Score the open coupon with everything the pipeline knows.

    Calibration is refitted here from silver's history rather than loading
    05's logged MLflow model. That keeps this notebook deterministic and
    free of a model-registry lookup; swapping to the registered model is a
    refinement, not a requirement.
    """
    matches = load_matches()

    models = fit_all_calibration_curves(to_long_format(matches))
    calibrated = calibrate_matches(matches, models)

    elo = build_elo(matches)

    draw_number = open_draw_number(matches)
    predictions = build_mart_coupon(calibrated, elo, draw_number)

    rows = predictions.collect()
    print(f"draw {draw_number} -- {len(rows)} matches")
    print()
    compact = show(rows)

    missing = [row["event_number"] for row in rows if row["elo_home"] is None]
    if missing:
        print()
        print(f"no Elo for event(s) {missing}: a team with no prior match in the data")

    write_coupon(predictions)
    print()
    print(f"written to {COUPON_TABLE}")

    dbutils.notebook.exit(
        f"draw={draw_number} matches={len(rows)} without_elo={len(missing)} || {compact}"
    )

main()